# HDX WFP food prices (Nigeria) → landing_dev (Stage 1)

Full CSV by default (`wfp_food_prices_nga.csv`). Set `PILOT = True` for a small head slice.

- Guide: [`hdx_wfp_food_prices_nigeria.md`](./hdx_wfp_food_prices_nigeria.md)
- Preferred env: `data-eng/venv`. First cell bootstraps `ingestion` if needed.
- Creds: `data-eng/ingestion/.env`

## Setup (landing_dev)

In [ ]:
import sys
from pathlib import Path


def _load_notebook_bootstrap() -> None:
    here = Path.cwd().resolve()
    candidates: list[Path] = []
    for base in [here, *here.parents]:
        candidates.extend(
            [
                base / "_bootstrap.py",
                base / "notebooks" / "_bootstrap.py",
                base / "ingestion" / "notebooks" / "_bootstrap.py",
                base / "data-eng" / "ingestion" / "notebooks" / "_bootstrap.py",
            ]
        )
    for boot in candidates:
        if boot.is_file():
            nb_dir = str(boot.parent)
            if nb_dir not in sys.path:
                sys.path.insert(0, nb_dir)
            from _bootstrap import ensure_ingestion

            print("ingestion ready", ensure_ingestion())
            return
    raise ModuleNotFoundError("Could not find notebooks/_bootstrap.py")


_load_notebook_bootstrap()

from ingestion.common.auth import ensure_auth_safe
from ingestion.common.bq import get_client
from ingestion.common.config import get_settings
from ingestion.common.load import add_metadata, load_dataframe, new_run_id

# Full CSV by default. Set True only for a small discovery slice.
PILOT = False
PILOT_MAX_ROWS = 500

TABLE = "hdx_wfp_food_prices_nigeria"
SOURCE_NAME = "hdx_wfp_food_prices_nigeria"
SOURCE_URL = (
    "https://data.humdata.org/dataset/42db041f-7aaf-4ab4-961f-2a12096861e7/"
    "resource/12b51155-0cd3-4806-9924-61ede4077591/download/wfp_food_prices_nga.csv"
)
KEY_COLS = ["date", "market_id", "commodity_id", "pricetype", "unit"]

settings = get_settings(cli_target="landing_dev")
ensure_auth_safe(settings)
run_id = new_run_id()
print("python", sys.executable)
print("mode", "PILOT" if PILOT else "FULL")
print("target", settings.target_dataset, "project", settings.project, "run_id", run_id)

## Fetch HDX CSV (`wfp_food_prices_nga.csv`)

Skip the HXL tag row (second line starting with `#`).

In [ ]:
from io import StringIO

import pandas as pd
import requests


def skip_hxl(text: str) -> str:
    lines = text.splitlines()
    if len(lines) >= 2 and lines[1].lstrip().startswith("#"):
        return "\n".join([lines[0], *lines[2:]])
    return text


resp = requests.get(SOURCE_URL, timeout=180)
resp.raise_for_status()
text = skip_hxl(resp.content.decode("utf-8", errors="replace"))
raw = pd.read_csv(StringIO(text))
if PILOT:
    raw = raw.iloc[:PILOT_MAX_ROWS].copy()
print("rows", len(raw), "cols", list(raw.columns))
raw.head()

## Normalize (same rules as package `hdx_wfp_prices` transform)

In [ ]:
TYPED = {
    "date": "datetime64[ns]",
    "market_id": "Int64",
    "commodity_id": "Int64",
    "price": "float64",
    "usdprice": "float64",
    "latitude": "float64",
    "longitude": "float64",
}

df = pd.DataFrame(raw)
df.columns = [str(c).strip().lower() for c in df.columns]
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
for col, dtype in TYPED.items():
    if col in df.columns and col != "date":
        df[col] = pd.to_numeric(df[col], errors="coerce").astype(dtype)
for col in ("market", "commodity", "pricetype", "unit", "currency"):
    if col in df.columns:
        df[col] = df[col].astype(str)
df = pd.DataFrame(df)
print("rows after normalize", len(df))
df.head()

## Preview / optional local parquet

In [ ]:
ready = add_metadata(
    pd.DataFrame(df),
    source=SOURCE_NAME,
    run_id=run_id,
    source_url=SOURCE_URL,
    key_cols=KEY_COLS,
)
settings.out_dir.mkdir(parents=True, exist_ok=True)
preview_path = settings.out_dir / f"{SOURCE_NAME}_{run_id}.parquet"
ready.to_parquet(preview_path, index=False)
print("preview", preview_path, "rows", len(ready))
ready.head()

## Load into landing_dev

`replace_partition` load job (not `insert_rows_json`).

In [ ]:
client = get_client(settings)
result = load_dataframe(
    client,
    settings,
    ready,
    table=TABLE,
    mode="replace_partition",
    run_id=run_id,
)
print(result)

## After review → Stage 2

Package already exists as CLI source **`hdx_wfp_prices`** (table `hdx_wfp_food_prices`).  
`python -m ingestion run hdx_wfp_prices` / `make ingest SOURCE=hdx_wfp_prices`  
Prod `landing` requires `INGEST_ALLOW_PROD=1` and `CI=true`.